# Xsurv: interpretable CpG markers for melanoma stage
A step-by-step analysis of the supplied **Xsurv.csv**. Outcome: low stage (original label 1) versus high stage (original label 2). We seek a compact predictive CpG panel and compare linear, bagged-tree and gradient-boosting classifiers; this is not a survival model.

## How to run
Place this notebook next to `Xsurv.csv`, open it in Jupyter or Google Colab, and run all cells in order. Alternatively set `DATA_PATH` below. Required packages: numpy, pandas, scipy, scikit-learn, matplotlib, seaborn, xgboost and lightgbm. If XGBoost or LightGBM is missing, the setup cell installs it with `pip`; internet access is needed only for that installation. The notebook records package versions. Tables and PNG figures are exported to `Xsurv_results`.

## Analysis map
1. Audit and provenance → 2. stratified holdout → 3. training-data exploration → 4. nested model comparison → 5. compact-panel and model locking → 6. stability and interpretation → 7. locked test evaluation → 8. annotation and exports.

**Scope:** These are candidate stage-associated markers. No external validation, survival analysis or causal inference is performed. The original filtering of the 197 CpGs and their transformation are unknown. A split performed now cannot undo possible leakage introduced before this CSV was created. Rows are assumed to be independent patients; patient, batch and site metadata were not provided.


In [ ]:
from pathlib import Path
import sys, json, hashlib, platform, warnings, subprocess, importlib.util

# Install optional gradient-boosting libraries only if they are missing.
for pkg in ['xgboost', 'lightgbm']:
    if importlib.util.find_spec(pkg) is None:
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import numpy as np
import pandas as pd
import scipy, sklearn, matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost
import lightgbm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score, recall_score,
                             confusion_matrix, roc_curve, precision_recall_curve,
                             average_precision_score, brier_score_loss)
from sklearn.exceptions import ConvergenceWarning

SEED = 42
DATA_PATH = None  # e.g. Path('/your/folder/Xsurv.csv')
OUT = Path('Xsurv_results')
OUT.mkdir(exist_ok=True)
OUTER_FOLDS, INNER_FOLDS = 5, 3
N_BOOTSTRAP = 2000
THRESHOLD = 0.5  # fixed in advance; never optimize on the test set
PANEL_TOLERANCE = 0.02  # descriptive AUC tolerance, not a noninferiority margin

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 180, 'font.size': 10})

versions = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scipy': scipy.__version__,
            'scikit-learn': sklearn.__version__, 'matplotlib': matplotlib.__version__,
            'seaborn': sns.__version__, 'xgboost': xgboost.__version__,
            'lightgbm': lightgbm.__version__}
print(json.dumps(versions, indent=2))

def table(obj):
    print(obj.to_string() if hasattr(obj, 'to_string') else obj)

def finish(name):
    plt.tight_layout()
    plt.savefig(OUT / (name + '.png'), bbox_inches='tight')
    plt.show()

# Do not silently accept a nonconverged logistic model.
warnings.filterwarnings('error', category=ConvergenceWarning)


## 1. Load and audit the dataset
The first CSV column is a row identifier, not a predictor. We verify the expected structure, missingness, duplicate records and numeric values. SEX remains an unnamed binary code because its category mapping is unavailable. AGE units are also unverified. We do not guess or remove unusual ages.

CpG values outside [0, 1] cannot be raw methylation beta values. Negative values alone do not tell us whether these are standardized values, M-values, or another transformation.

In [ ]:
if DATA_PATH is None:
    candidates = [Path('Xsurv.csv'), Path('../data/Xsurv.csv'), Path('../upload/Xsurv.csv')]
    DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None or not Path(DATA_PATH).exists():
    raise FileNotFoundError('Set DATA_PATH to the supplied Xsurv.csv.')
DATA_PATH = Path(DATA_PATH)
df = pd.read_csv(DATA_PATH, index_col=0)
cpgs = [c for c in df if c.startswith('cg')]
clinical = ['AGE', 'SEX']
assert {'AGE', 'SEX', 'Stage'}.issubset(df.columns)
assert set(df.Stage.unique()) == {1, 2}
assert df.index.is_unique and df.columns.is_unique
assert df.select_dtypes(include='number').shape[1] == df.shape[1]
assert np.isfinite(df.to_numpy()).all(), 'Missing/nonfinite values need a training-only imputer.'
assert not df.duplicated().any(), 'Review duplicates before splitting.'
audit = pd.Series({'rows': len(df), 'variables': df.shape[1], 'CpGs': len(cpgs),
                   'missing_values': int(df.isna().sum().sum()),
                   'constant_columns': int((df.nunique() <= 1).sum()),
                   'AGE_min': df.AGE.min(), 'AGE_max': df.AGE.max(),
                   'CpG_min': df[cpgs].min().min(), 'CpG_max': df[cpgs].max().max()})
table(audit)
table(df.iloc[:5, :7])
print('SEX codes:', sorted(df.SEX.unique()))
print('Dataset SHA256:', hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())

## 2. Reserve the test set before exploration
We recode low stage to 0 and high stage to 1, so all positive-class metrics refer to high stage. The 80/20 split is stratified. Test labels are used here only to preserve and report class counts. Test predictors are not explored or fitted until the final evaluation.

This is an internal held-out test set from the same dataset, not an independent external cohort.

In [ ]:
X = df[clinical + cpgs].copy()
y = df.Stage.map({1: 0, 2: 1}).astype(int)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
assert set(X_dev.index).isdisjoint(X_test.index)
counts = pd.DataFrame({'Development': y_dev.value_counts(), 'Test': y_test.value_counts()})
counts = counts.reindex([0, 1])
counts.index = ['Low stage', 'High stage']
table(counts)
counts.T.plot(kind='bar', color=['#377eb8', '#e37742'], rot=0, figsize=(7, 4))
plt.ylabel('Patients'); plt.title('Stratified development/test split')
finish('01_split')
split_manifest = pd.DataFrame({'row_id': df.index, 'partition': np.where(df.index.isin(X_dev.index), 'development', 'test')})

## 3. Explore development patients only
The following plots describe the development set. A PCA separation is exploratory and is not a predictive-performance estimate. PCA uses CpGs only, so age and sex cannot dominate the axes. Its fitted scaler and components are not reused by the classifiers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
sns.histplot(data=X_dev.assign(Stage=y_dev.map({0:'Low', 1:'High'})), x='AGE', hue='Stage', bins=15, ax=axes[0])
axes[0].set_title('AGE as supplied; units unverified')
sns.countplot(data=X_dev.assign(Stage=y_dev.map({0:'Low', 1:'High'})), x='SEX', hue='Stage', ax=axes[1])
axes[1].set_title('SEX codes; mapping unverified')
axes[2].hist(X_dev[cpgs].to_numpy().ravel(), bins=50, color='#498b99')
axes[2].set(title='Supplied CpG value distribution', xlabel='Transformed value', ylabel='Measurements')
finish('02_distributions')

corr = X_dev[cpgs].corr()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
high_corr = upper[upper.abs() > 0.9].rename('Pearson_r').reset_index()
high_corr.columns = ['CpG_1', 'CpG_2', 'Pearson_r']
print('Development-only pairs with |r| > 0.9:'); table(high_corr)
plt.figure(figsize=(7, 6))
sns.heatmap(corr, cmap='vlag', center=0, vmin=-1, vmax=1, xticklabels=False, yticklabels=False)
plt.title('Development CpG correlations'); plt.xlabel('197 CpGs'); plt.ylabel('197 CpGs')
finish('03_correlations')

pca = PCA()
z = StandardScaler().fit_transform(X_dev[cpgs])
pcs = pca.fit_transform(z)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(1, len(pca.explained_variance_ratio_)+1), np.cumsum(pca.explained_variance_ratio_))
axes[0].set(xlabel='Number of components', ylabel='Cumulative explained variance', title='PCA variance')
sns.scatterplot(x=pcs[:,0], y=pcs[:,1], hue=y_dev.map({0:'Low',1:'High'}).to_numpy(), ax=axes[1], alpha=0.8)
axes[1].set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', title='Development patients')
finish('04_pca')

## 4. Define models and nested validation
All models share the same five outer folds. Each outer training set gets a separate three-fold inner grid search. Any scaling or CpG selection stays inside the pipeline and is refitted within each inner fold.

We compare four algorithm families across **clinical-only**, **CpG-only** and **combined** predictors:

- **Elastic-net logistic regression:** a regularized linear model suited to many correlated predictors. It can shrink weak CpG coefficients toward zero.
- **Random forest:** an ensemble of decision trees. It provides a nonlinear comparison and can model interactions without feature scaling.
- **XGBoost:** regularized sequential gradient-boosted trees. It can capture nonlinear effects and interactions while correcting errors made by earlier trees.
- **LightGBM:** histogram-based gradient-boosted trees with efficient leaf-wise growth. It provides another nonlinear boosting approach and is computationally efficient for tabular predictors.

For compact CpG panels, `SelectKBest` ranks CpGs using training-only ANOVA F scores, then an L2 logistic model combines them. Panel sizes 5, 10 and 20 are fixed before evaluation.

The hyperparameter grids are deliberately small because the development cohort is modest. This keeps nested CV tractable and reduces the risk of searching a very large tuning space. The analysis is a controlled algorithm comparison, not an exhaustive benchmark.

**Primary metric:** ROC-AUC.  
**Secondary metrics:** balanced accuracy, high-stage sensitivity and low-stage specificity at the fixed threshold 0.5.

ROC-AUC measures ranking across thresholds, while the threshold-based metrics show classification behavior at the prespecified decision threshold. Fold spread is descriptive and is not a confidence interval.


In [ ]:
def elastic_model():
    # scikit-learn >= 1.8 expresses elastic-net through l1_ratio.
    legacy = {'penalty': 'elasticnet'} if tuple(map(int, sklearn.__version__.split('.')[:2])) < (1, 8) else {}
    return Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(**legacy, solver='saga',
                                     max_iter=20000, tol=1e-3, random_state=SEED))
    ])

def forest_model():
    return Pipeline([
        ('model', RandomForestClassifier(
            n_estimators=150, random_state=SEED, n_jobs=1
        ))
    ])

def xgb_model():
    return Pipeline([
        ('model', XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            n_estimators=200,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            random_state=SEED,
            n_jobs=1,
            tree_method='hist',
            verbosity=0
        ))
    ])

def lightgbm_model():
    return Pipeline([
        ('model', LGBMClassifier(
            objective='binary',
            n_estimators=200,
            min_child_samples=10,
            colsample_bytree=0.8,
            random_state=SEED,
            n_jobs=1,
            verbosity=-1
        ))
    ])

def panel_model(k):
    return Pipeline([
        ('scale', StandardScaler()),
        ('select', SelectKBest(f_classif, k=k)),
        ('model', LogisticRegression(solver='lbfgs', max_iter=5000, random_state=SEED))
    ])

feature_sets = {'Clinical': clinical, 'CpG': cpgs, 'Combined': clinical + cpgs}
specs = {}

for label, cols in feature_sets.items():
    specs[label + ' elastic-net'] = (
        cols,
        elastic_model(),
        {'model__C': [0.03, 0.3, 3],
         'model__l1_ratio': [0.25, 0.75]}
    )
    specs[label + ' forest'] = (
        cols,
        forest_model(),
        {'model__max_depth': [3, None],
         'model__min_samples_leaf': [3, 8]}
    )
    specs[label + ' XGBoost'] = (
        cols,
        xgb_model(),
        {'model__max_depth': [2, 3],
         'model__learning_rate': [0.03, 0.1]}
    )
    specs[label + ' LightGBM'] = (
        cols,
        lightgbm_model(),
        {'model__num_leaves': [7, 15],
         'model__learning_rate': [0.03, 0.1]}
    )

for k in [5, 10, 20]:
    specs[f'Panel {k}'] = (
        cpgs,
        panel_model(k),
        {'model__C': [0.03, 0.3, 3]}
    )

outer = list(StratifiedKFold(
    OUTER_FOLDS, shuffle=True, random_state=SEED
).split(X_dev, y_dev))

def fit_search(name, xx, yy, seed):
    cols, estimator, grid = specs[name]
    cv = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed)
    search = GridSearchCV(
        clone(estimator), grid, scoring='roc_auc', cv=cv, n_jobs=1,
        refit=True, error_score='raise'
    )
    search.fit(xx[cols], yy)
    return search

print(f'{len(specs)} candidate model configurations defined:')
print('\n'.join(specs.keys()))


def metrics(yy, probability):
    pred = np.asarray(probability) >= THRESHOLD
    return {
        'ROC_AUC': roc_auc_score(yy, probability),
        'Balanced_accuracy': balanced_accuracy_score(yy, pred),
        'Sensitivity': recall_score(yy, pred, pos_label=1, zero_division=0),
        'Specificity': recall_score(yy, pred, pos_label=0, zero_division=0)
    }


### Run nested cross-validation
This is the main computational step. Each reported outer-fold prediction comes from a model whose preprocessing, marker selection and tuning never saw that outer validation fold. We retain fitted outer models for descriptive marker stability.

Selecting the best family from these outer results can itself make the winning cross-validation score optimistic. The untouched test set provides the final assessment of the locked choices.

In [ ]:
records, outer_models, oof = [], {}, {}
for name in specs:
    outer_models[name] = []
    oof[name] = np.full(len(y_dev), np.nan)
    for fold, (tr, va) in enumerate(outer, 1):
        search = fit_search(name, X_dev.iloc[tr], y_dev.iloc[tr], SEED+fold)
        model = search.best_estimator_
        prob = model.predict_proba(X_dev.iloc[va][specs[name][0]])[:,1]
        oof[name][va] = prob
        records.append({'Model':name, 'Fold':fold, **metrics(y_dev.iloc[va],prob),
                        'Inner_best_AUC':search.best_score_, 'Parameters':json.dumps(search.best_params_)})
        outer_models[name].append(model)
    print(name, 'completed', flush=True)
cv_results = pd.DataFrame(records)
summary = cv_results.groupby('Model').agg(Mean_AUC=('ROC_AUC','mean'), SD_AUC=('ROC_AUC','std'),
           Mean_balanced_accuracy=('Balanced_accuracy','mean'),
           Mean_sensitivity=('Sensitivity','mean'), Mean_specificity=('Specificity','mean'))
summary = summary.sort_values('Mean_AUC',ascending=False)
table(summary.round(3))

In [ ]:
order = summary.index.tolist()
fig, ax = plt.subplots(figsize=(10, 7))
sns.stripplot(data=cv_results, x='ROC_AUC', y='Model', order=order,
              jitter=False, color='#5f929c', size=6, ax=ax)
ax.scatter(summary.Mean_AUC, np.arange(len(order)), marker='D',
           color='#c34432', label='Mean of five folds', zorder=4)
ax.axvline(0.5, color='gray', linestyle='--')
ax.set_xlim(0, 1)
ax.set_title('Nested CV: dots are outer folds; diamonds are means')
ax.legend(loc='lower left')
finish('05_nested_comparison')

# Direct algorithm comparison across predictor sets.
algorithm_order = ['elastic-net', 'forest', 'XGBoost', 'LightGBM']
heat = pd.DataFrame(
    index=['Clinical', 'CpG', 'Combined'],
    columns=algorithm_order,
    dtype=float
)
for fs in heat.index:
    for alg in heat.columns:
        heat.loc[fs, alg] = summary.loc[f'{fs} {alg}', 'Mean_AUC']

plt.figure(figsize=(8, 4))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='viridis', vmin=0, vmax=1)
plt.title('Mean outer-fold ROC-AUC by predictor set and algorithm')
plt.xlabel('Algorithm')
plt.ylabel('Predictor set')
finish('05b_algorithm_feature_heatmap')

fig, ax = plt.subplots(figsize=(7, 4))
for name in ['Panel 5', 'Panel 10', 'Panel 20', 'CpG elastic-net']:
    s = cv_results.loc[cv_results.Model.eq(name), 'ROC_AUC']
    ax.errorbar(name, s.mean(), yerr=s.std(), fmt='o', capsize=5, color='#377e91')
ax.set(ylabel='Outer-fold ROC-AUC',
       title='Panel size comparison: mean ± fold SD',
       ylim=(0, 1))
finish('06_panel_sizes')


## 5. Lock the analysis choices before test evaluation
Choose the smallest CpG panel within 0.02 mean AUC of the best compact panel. This practical rule is fixed above and does not establish statistical equivalence.

Before touching the test set, also lock:

1. the highest-AUC model overall,
2. the strongest clinical-only baseline,
3. the strongest full-CpG model,
4. the best XGBoost configuration from development CV, and
5. the best LightGBM configuration from development CV.

Including the best XGBoost and LightGBM configurations ensures that both boosting approaches receive a prespecified held-out evaluation even if neither is the overall development-CV winner. Duplicate roles are removed automatically.

Test results must not be used to choose a new winner, revise hyperparameters, change the CpG panel or optimize the probability threshold. All held-out comparisons are descriptive unless formally tested in an independent design.


In [ ]:
panel_names = ['Panel 5', 'Panel 10', 'Panel 20']
best_panel_auc = summary.loc[panel_names, 'Mean_AUC'].max()
selected_panel_name = next(
    n for n in panel_names
    if summary.loc[n, 'Mean_AUC'] >= best_panel_auc - PANEL_TOLERANCE
)

winner_name = summary.index[0]

clinical_candidates = [
    'Clinical elastic-net', 'Clinical forest',
    'Clinical XGBoost', 'Clinical LightGBM'
]
clinical_name = summary.loc[clinical_candidates, 'Mean_AUC'].idxmax()

full_candidates = [
    'CpG elastic-net', 'CpG forest',
    'CpG XGBoost', 'CpG LightGBM'
]
full_name = summary.loc[full_candidates, 'Mean_AUC'].idxmax()

xgb_candidates = [n for n in summary.index if n.endswith('XGBoost')]
lgbm_candidates = [n for n in summary.index if n.endswith('LightGBM')]
xgb_name = summary.loc[xgb_candidates, 'Mean_AUC'].idxmax()
lgbm_name = summary.loc[lgbm_candidates, 'Mean_AUC'].idxmax()

locked_names = list(dict.fromkeys([
    selected_panel_name,
    winner_name,
    clinical_name,
    full_name,
    xgb_name,
    lgbm_name
]))

print('Compact panel:', selected_panel_name)
print('Overall model:', winner_name)
print('Clinical comparator:', clinical_name)
print('Full-CpG comparator:', full_name)
print('Best XGBoost comparator:', xgb_name)
print('Best LightGBM comparator:', lgbm_name)

final_searches = {
    name: fit_search(name, X_dev, y_dev, SEED + 100)
    for name in locked_names
}
final_models = {
    name: s.best_estimator_
    for name, s in final_searches.items()
}

final_panel = final_models[selected_panel_name]
panel_genes = np.array(cpgs)[
    final_panel.named_steps['select'].get_support()
]

print('Locked CpGs:', ', '.join(panel_genes))
print('Locked threshold:', THRESHOLD)
print('Locked models:', ', '.join(locked_names))


## 6. Marker stability and interpretation
Selection frequencies below are fractions of five overlapping outer training sets, not probabilities of biological truth. Panel frequency measures training-only F-score selection; elastic-net frequency measures nonzero coefficients. Elastic-net sign consistency is the larger of positive/negative counts divided by the nonzero count. A value of 1 means all nonzero fits had the same sign.

Final panel coefficients represent a one-development-set-SD increase in the supplied transformed CpG value, conditional on other panel markers. Positive coefficients favor high stage. They do not imply increased raw methylation or causal effects. Descriptive group differences reuse the development data and are not independent validation.

In [ ]:
panel_selected = np.array([m.named_steps['select'].get_support() for m in outer_models[selected_panel_name]])
enet_coef = np.array([m.named_steps['model'].coef_[0] for m in outer_models['CpG elastic-net']])
nonzero = np.abs(enet_coef)>1e-8
n_selected = nonzero.sum(axis=0)
consistent = np.divide(np.maximum((enet_coef>1e-8).sum(axis=0),(enet_coef < -1e-8).sum(axis=0)),
                       n_selected,out=np.full(len(cpgs),np.nan),where=n_selected>0)
marker_ranking = pd.DataFrame({'CpG':cpgs, 'Panel_selection_frequency':panel_selected.mean(axis=0),
    'Elastic_net_selection_frequency':nonzero.mean(axis=0), 'Elastic_net_sign_consistency':consistent,
    'Mean_elastic_net_coefficient':enet_coef.mean(axis=0)})
marker_ranking['In_final_panel'] = marker_ranking.CpG.isin(panel_genes)
marker_ranking = marker_ranking.sort_values(['Panel_selection_frequency','Elastic_net_selection_frequency'],ascending=False)
final_panel_table = pd.DataFrame({'CpG':panel_genes,'Standardized_coefficient':final_panel.named_steps['model'].coef_[0]})
final_panel_table = final_panel_table.merge(marker_ranking,on='CpG',validate='one_to_one')
final_panel_table['Development_mean_low'] = [X_dev.loc[y_dev.eq(0),c].mean() for c in final_panel_table.CpG]
final_panel_table['Development_mean_high'] = [X_dev.loc[y_dev.eq(1),c].mean() for c in final_panel_table.CpG]
table(final_panel_table.round(3))
fig, axes = plt.subplots(1,2,figsize=(13,6))
s = marker_ranking.head(15).set_index('CpG')[['Panel_selection_frequency','Elastic_net_selection_frequency']]
s.iloc[::-1].plot.barh(ax=axes[0],color=['#377eb8','#e37742'])
axes[0].set(xlim=(0,1),title='Selection frequency across five outer fits',xlabel='Fraction of fits')
axes[0].legend(['Panel','Elastic net'],fontsize=8)
c = final_panel_table.sort_values('Standardized_coefficient')
axes[1].barh(c.CpG,c.Standardized_coefficient,color=np.where(c.Standardized_coefficient>0,'#e37742','#377eb8'))
axes[1].axvline(0,color='black',linewidth=0.7); axes[1].set(title='Final panel coefficients',xlabel='Coefficient per training SD')
finish('07_marker_stability')

plot_markers = final_panel_table.reindex(final_panel_table.Standardized_coefficient.abs().sort_values(ascending=False).index).CpG.head(6).tolist()
fig, axes = plt.subplots(2,3,figsize=(12,7)); axes=axes.ravel()
for ax, marker in zip(axes,plot_markers):
    sns.boxplot(x=y_dev.map({0:'Low',1:'High'}),y=X_dev[marker],ax=ax,color='#a9cad1',fliersize=2)
    ax.set(title=marker,xlabel='Stage',ylabel='Supplied CpG value')
for ax in axes[len(plot_markers):]: ax.axis('off')
finish('08_marker_distributions')

## 7. Final held-out evaluation
Everything above is now fixed. We generate test probabilities once per locked model and calculate metrics at threshold 0.5.

The 95% percentile intervals use 2,000 stratified bootstrap samples of test patients, preserving the observed low/high counts. They quantify test-sample uncertainty conditional on fitted models. They do not include model-selection, training or original preprocessing uncertainty. With only 64 test patients, conclusions may remain imprecise.

The clinical and full-CpG comparisons are descriptive. We do not claim a significant improvement from point estimates alone. Probability calibration is summarized by the Brier score, but not recalibrated on test data.

In [ ]:
test_prob = {name:model.predict_proba(X_test[specs[name][0]])[:,1] for name,model in final_models.items()}
test_point = pd.DataFrame({name:{**metrics(y_test,p),'Average_precision':average_precision_score(y_test,p),
                                'Brier_score':brier_score_loss(y_test,p)} for name,p in test_prob.items()}).T
table(test_point.round(3))
rng = np.random.default_rng(SEED)
y_arr = y_test.to_numpy()
low_idx, high_idx = np.flatnonzero(y_arr==0), np.flatnonzero(y_arr==1)
boot_idx = [np.concatenate([rng.choice(low_idx,len(low_idx),replace=True),rng.choice(high_idx,len(high_idx),replace=True)]) for _ in range(N_BOOTSTRAP)]
interval_records=[]
bootstrap_auc = {}
for name,p in test_prob.items():
    samples = pd.DataFrame([metrics(y_arr[ix],p[ix]) for ix in boot_idx])
    bootstrap_auc[name] = samples.ROC_AUC.to_numpy()
    for metric in samples:
        lo,hi = np.quantile(samples[metric],[0.025,0.975])
        interval_records.append({'Model':name,'Metric':metric,'Estimate':test_point.loc[name,metric], 'Lower_95':lo,'Upper_95':hi})
test_intervals = pd.DataFrame(interval_records)
table(test_intervals.round(3))
delta = bootstrap_auc[selected_panel_name]-bootstrap_auc[clinical_name]
print('Panel minus clinical AUC:', round(test_point.loc[selected_panel_name,'ROC_AUC']-test_point.loc[clinical_name,'ROC_AUC'],3))
print('Paired bootstrap 95% interval:',np.round(np.quantile(delta,[0.025,0.975]),3))

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,4.6))
for name,p in test_prob.items():
    fpr,tpr,_ = roc_curve(y_test,p)
    axes[0].plot(fpr,tpr,label=f'{name}: {roc_auc_score(y_test,p):.2f}')
    precision,recall,_ = precision_recall_curve(y_test,p)
    axes[1].plot(recall,precision,label=name)
axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set(xlabel='False-positive rate',ylabel='True-positive rate',title='Held-out ROC curves')
axes[0].legend(fontsize=8)
axes[1].axhline(y_test.mean(),color='gray',linestyle='--')
axes[1].set(xlabel='Recall (high stage)',ylabel='Precision (high stage)',title='Held-out precision–recall')
axes[1].legend(fontsize=8)
cm = confusion_matrix(y_test,test_prob[selected_panel_name]>=THRESHOLD,labels=[0,1])
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',cbar=False,xticklabels=['Low','High'],yticklabels=['Low','High'],ax=axes[2])
axes[2].set(xlabel='Predicted stage',ylabel='Observed stage',title=f'{selected_panel_name}: threshold {THRESHOLD}')
finish('09_test_performance')

plt.figure(figsize=(8,4.5))
ci = test_intervals.query("Metric == 'ROC_AUC'").set_index('Model')
plt.errorbar(ci.Estimate, np.arange(len(ci)), xerr=[ci.Estimate-ci.Lower_95,ci.Upper_95-ci.Estimate],fmt='o',capsize=4)
plt.yticks(np.arange(len(ci)),ci.index); plt.axvline(0.5,color='gray',linestyle='--')
plt.xlim(0,1); plt.xlabel('ROC-AUC with conditional 95% bootstrap interval')
plt.title('Final test estimates and uncertainty')
finish('10_test_intervals')

## 8. Biological annotation without invented mappings
The CSV contains probe IDs but no assay platform, genome build, gene mapping or genomic coordinates. Automatic gene assignment would be unreliable without a matching annotation manifest. Therefore this section is executable but conditional: it merges a supplied, verified annotation CSV if present; otherwise it exports an explicitly unresolved annotation table.

To complete this step, confirm the assay and obtain its matching manifest from the original data provider or manufacturer. Prepare `CpG_annotation.csv` with one row per CpG and these columns: `CpG`, `Gene`, `Chromosome`, `Position`, `Genome_build`, `Annotation_source`. Preserve multiple mapped genes within the Gene field. A nearest-gene label is not proof that a CpG regulates that gene. Do not infer methylation direction from the coefficient until the value transformation is documented.

In [ ]:
ANNOTATION_PATH = Path('CpG_annotation.csv')
annotation_columns = ['Gene','Chromosome','Position','Genome_build','Annotation_source']
if ANNOTATION_PATH.exists():
    annotation = pd.read_csv(ANNOTATION_PATH,dtype=str)
    assert set(['CpG']+annotation_columns).issubset(annotation.columns)
    assert annotation.CpG.is_unique, 'Consolidate multiple mappings into one row per CpG.'
    annotated_panel = final_panel_table.merge(annotation[['CpG']+annotation_columns],on='CpG',how='left',validate='one_to_one')
    annotated_panel['Annotation_status'] = np.where(annotated_panel.Annotation_source.notna(),'Provided mapping; verify source','Unresolved')
else:
    annotated_panel = final_panel_table.copy()
    for c in annotation_columns: annotated_panel[c] = pd.NA
    annotated_panel['Annotation_status'] = 'Unresolved: platform-matched annotation required'
print('Annotation status:'); table(annotated_panel[['CpG','Gene','Annotation_status']])

## 9. Export results and summarize the evidence
Exports include the split manifest, fold-level results, full marker ranking, locked panel, test predictions and uncertainty intervals. Re-running preserves the seed but software-version changes can affect results. A separate external cohort is required to assess transportability. Do not rerun with different seeds until a favorable test result appears.

Use the generated conclusion below to report what this run actually found. A high internal score does not resolve unknown upstream filtering, batch effects, independence or clinical utility.

In [ ]:
outputs = {'data_audit': audit.rename('value').to_frame(),
           'split_manifest': split_manifest,
           'high_correlations_development': high_corr,
           'nested_cv_folds': cv_results,
           'nested_cv_summary': summary,
           'marker_ranking': marker_ranking,
           'final_panel': annotated_panel,
           'test_metrics': test_point,
           'test_intervals': test_intervals}

for name, frame in outputs.items():
    frame.to_csv(
        OUT / (name + '.csv'),
        index=name in ['data_audit', 'nested_cv_summary', 'test_metrics']
    )

predictions = pd.DataFrame({
    'row_id': X_test.index,
    'Stage_original': y_test.to_numpy() + 1
})
for name, p in test_prob.items():
    predictions[name + '_prob_high'] = p
predictions.to_csv(OUT / 'test_predictions.csv', index=False)

manifest = {
    'seed': SEED,
    'test_fraction': 0.2,
    'threshold': THRESHOLD,
    'outer_folds': OUTER_FOLDS,
    'inner_folds': INNER_FOLDS,
    'panel_tolerance': PANEL_TOLERANCE,
    'selected_panel': selected_panel_name,
    'winner': winner_name,
    'clinical_comparator': clinical_name,
    'full_CpG_comparator': full_name,
    'best_XGBoost_comparator': xgb_name,
    'best_LightGBM_comparator': lgbm_name,
    'locked_models': locked_names,
    'selected_CpGs': panel_genes.tolist(),
    'parameters': {n: s.best_params_ for n, s in final_searches.items()},
    'versions': versions,
    'input_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
}
(OUT / 'analysis_manifest.json').write_text(json.dumps(manifest, indent=2))

r = test_intervals.query(
    "Model == @selected_panel_name and Metric == 'ROC_AUC'"
).iloc[0]

print(f'Locked panel: {selected_panel_name}; {len(panel_genes)} CpGs.')
print(
    f'Internal held-out AUC: {r.Estimate:.3f} '
    f'(conditional bootstrap 95% interval '
    f'{r.Lower_95:.3f}–{r.Upper_95:.3f}).'
)
print(f'Clinical comparator AUC: {test_point.loc[clinical_name, "ROC_AUC"]:.3f}.')
print(f'Full-CpG comparator AUC: {test_point.loc[full_name, "ROC_AUC"]:.3f}.')
print(f'Best XGBoost comparator ({xgb_name}) AUC: {test_point.loc[xgb_name, "ROC_AUC"]:.3f}.')
print(f'Best LightGBM comparator ({lgbm_name}) AUC: {test_point.loc[lgbm_name, "ROC_AUC"]:.3f}.')
print('This run provides candidate stage markers and internal predictive evidence only.')
print('Original CpG filtering, value transformation and biological annotation still need verification.')
print('Export directory:', OUT.resolve())


## References and interpretation guide
- Input data: user-supplied `Xsurv.csv`; starting specification: `project_Xsurv.ipynb`. The original notebook cites [Efficient gradient boosting for prognostic biomarker discovery](https://doi.org/10.1093/bioinformatics/btab869). This notebook analyzes the supplied stage outcome only; it does not reproduce that survival study.
- [scikit-learn: common pitfalls and data leakage](https://scikit-learn.org/stable/common_pitfalls.html): fit preprocessing and feature selection using training partitions only.
- [scikit-learn: nested versus non-nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html): separate tuning from outer-fold assessment.
- [XGBoost documentation](https://xgboost.readthedocs.io/): implementation used for regularized gradient-boosted trees.
- [LightGBM documentation](https://lightgbm.readthedocs.io/): implementation used for histogram-based gradient boosting.

**Reading the figures:** PCA shows major variation, not validated classification. Fold dots show sensitivity to the training split. The algorithm heatmap compares development-CV mean AUC across predictor sets and model families. Selection frequency shows reproducibility within development folds. Coefficient signs describe the fitted conditional association. ROC/precision–recall curves evaluate the locked models, including prespecified XGBoost and LightGBM comparators, on held-out patients. Bootstrap intervals describe uncertainty in those test patients, conditional on the trained models.

**Before a scientific claim:** verify independent patients, original filtering and normalization, AGE/SEX metadata, assay annotations and external replication. A compact panel or boosting model that performs poorly is still an informative result; do not relabel it as a validated biomarker model.
